# Feature engineering

Uses generated synthetic prices for software research only; these are not actual market data.

Warm-up values remain missing. This notebook also checks causal feature behavior.


In [ ]:
from pathlib import Path
import sys
repo = Path.cwd()
while not (repo / "backend").is_dir() and repo != repo.parent:
    repo = repo.parent
sys.path.insert(0, str(repo / "backend"))

from pandas.testing import assert_frame_equal
from trading.data.synthetic import generate_synthetic_ohlcv
from trading.features.feature_pipeline import engineer_features
bars = generate_synthetic_ohlcv(periods=600, frequency="1h", seed=11)
features = engineer_features(bars)
columns = [name for name in features if name not in bars]
features[["timestamp", "close", "return_5", "sma_20", "rsi_14",
          "macd_line", "atr_14", "rolling_volatility_20"]].tail()


In [ ]:
# Causality check: changing future OHLC bars must not change earlier features.
cutoff = 400
changed = bars.copy()
changed.loc[cutoff:, ["open", "high", "low", "close"]] *= 1.1
changed_features = engineer_features(changed)
assert_frame_equal(features.loc[:cutoff - 1, columns],
                   changed_features.loc[:cutoff - 1, columns],
                   check_exact=False, atol=1e-10)
print("Earlier features are unchanged by future bars.")
